# LightGBM Learning-to-Rank with LambdaRank

This notebook demonstrates how to build a learning-to-rank pipeline using `lgb.LGBMRanker` with the `lambdarank` objective, which optimises NDCG directly.

We use the **MovieLens 100k** dataset: each user is treated as a *query*, and their rated movies are the *candidate items*. The rating (1–5) serves as the graded relevance label.

```python
import lightgbm as lgb

ranker = lgb.LGBMRanker(objective="lambdarank", metric="ndcg", ...)
ranker.fit(train_x, train_y, group=train_group,
           eval_set=[(valid_x, valid_y)], eval_group=[valid_group],
           callbacks=[lgb.early_stopping(20)])
scores = ranker.predict(test_x)
ranker.booster_.save_model("model.lgb")

booster = lgb.Booster(model_file="model.lgb")
scores  = booster.predict(test_x)
```

## 0. Imports and Parameters

In [ ]:
import os
import sys
import lightgbm as lgb
import numpy as np
import pandas as pd
from tempfile import TemporaryDirectory
from sklearn.metrics import ndcg_score as sklearn_ndcg

try:
    from recommenders.models.lightgbm.lightgbm_utils import NumEncoder
    from recommenders.datasets import movielens
    from recommenders.utils.notebook_utils import store_metadata
except ImportError:
    # Force local project code by clearing cached recommenders modules first
    for key in list(sys.modules.keys()):
        if "recommenders" in key:
            del sys.modules[key]
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..")))

    from recommenders.models.lightgbm.lightgbm_utils import NumEncoder
    from recommenders.datasets import movielens
    from recommenders.utils.notebook_utils import store_metadata

print(f"Python: {sys.version}")

In [ ]:
# Data settings
SIZE       = "100k"
USER_COL   = "userID"
ITEM_COL   = "itemID"
RATING_COL = "rating"
LABEL_COL  = RATING_COL   # rating (1–5) as graded relevance label

# Engineered feature columns
NUME_COLS = ["user_mean_rating", "user_rating_count", "item_mean_rating", "item_rating_count"]
CATE_COLS = ["main_genre"]

# lgb.LGBMRanker parameters (lambdarank)
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5, 10],
    "num_leaves": 64,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "n_jobs": 4,
}
NUM_BOOST_ROUND       = 200
EARLY_STOPPING_ROUNDS = 20

## 1. Data Preparation

We use **MovieLens 100k**, which provides real user–item interactions with ratings 1–5.

**LTR framing:**
- **Query** = user (each user represents a search session)
- **Documents** = movies the user has rated
- **Relevance** = rating (1–5)

**Feature engineering** (computed from training data only to avoid leakage):
| Feature | Description |
|---|---|
| `user_mean_rating` | Average rating given by this user |
| `user_rating_count` | Number of ratings by this user |
| `item_mean_rating` | Average rating received by this movie |
| `item_rating_count` | Number of ratings for this movie (popularity) |
| `main_genre` | Primary genre of the movie (categorical) |

**Query groups** are formed by grouping interactions by `userID` — no simulation needed.

In [3]:
with TemporaryDirectory() as tmp:
    ratings = movielens.load_pandas_df(size=SIZE, local_cache_path=tmp)
    items   = movielens.load_item_df(size=SIZE, local_cache_path=tmp, genres_col="genre")

# Extract primary genre (first listed)
items["main_genre"] = items["genre"].str.split("|").str[0]
items = items[[ITEM_COL, "main_genre"]]

# 80 / 10 / 10 split (rows are already time-sorted in ML-100k)
n = len(ratings)
train_raw = ratings.iloc[: int(0.8 * n)].copy()
valid_raw = ratings.iloc[int(0.8 * n) : int(0.9 * n)].copy()
test_raw  = ratings.iloc[int(0.9 * n) :].copy()

# Compute user & item statistics from training data only (no label leakage)
user_stats = (
    train_raw.groupby(USER_COL)[RATING_COL]
    .agg(user_mean_rating="mean", user_rating_count="count")
    .reset_index()
)
item_stats = (
    train_raw.groupby(ITEM_COL)[RATING_COL]
    .agg(item_mean_rating="mean", item_rating_count="count")
    .reset_index()
)
global_mean = train_raw[RATING_COL].mean()

def enrich(df):
    df = df.merge(user_stats, on=USER_COL, how="left")
    df = df.merge(item_stats, on=ITEM_COL, how="left")
    df = df.merge(items, on=ITEM_COL, how="left")
    df["user_mean_rating"]  = df["user_mean_rating"].fillna(global_mean)
    df["user_rating_count"] = df["user_rating_count"].fillna(0)
    df["item_mean_rating"]  = df["item_mean_rating"].fillna(global_mean)
    df["item_rating_count"] = df["item_rating_count"].fillna(0)
    df["main_genre"]        = df["main_genre"].fillna("Unknown")
    return df

train_raw = enrich(train_raw)
valid_raw = enrich(valid_raw)
test_raw  = enrich(test_raw)

# Sort by user to form contiguous query groups
def make_groups(df):
    df = df.sort_values(USER_COL).reset_index(drop=True)
    group_sizes = df.groupby(USER_COL, sort=False).size().values
    return df, group_sizes

train_data, train_group = make_groups(train_raw)
valid_data, valid_group = make_groups(valid_raw)
test_data,  test_group  = make_groups(test_raw)

print(f"train: {len(train_data):,}  valid: {len(valid_data):,}  test: {len(test_data):,}")
print(f"train queries (users): {len(train_group):,}  valid: {len(valid_group):,}  test: {len(test_group):,}")

100%|██████████| 4.81k/4.81k [00:01<00:00, 3.00kKB/s]


train: 80,000  valid: 10,000  test: 10,000
train queries (users): 943  valid: 880  test: 878


## 2. Feature Encoding (NumEncoder)

`NumEncoder` converts all categorical columns into dense numerical features through three steps:
1. **Low-frequency filtering** — rare categories become `<LESS>`, missing values become `<UNK>`.
2. **Sequential target & count encoding** — encodes each sample using statistics from previous samples only (no label leakage).
3. **Binary encoding** — the ordinal-encoded category values are expanded into bit vectors.

In [4]:
encoder = NumEncoder(CATE_COLS, NUME_COLS, LABEL_COL)

train_x, train_y = encoder.fit_transform(train_data)
valid_x, valid_y = encoder.transform(valid_data)
test_x,  test_y  = encoder.transform(test_data)

print(f"train_x: {train_x.shape}  valid_x: {valid_x.shape}  test_x: {test_x.shape}")

2026-03-08 15:59:20,961 [INFO] Filtering and fillna features
100%|██████████| 4/4 [00:00<00:00, 2791.09it/s]
2026-03-08 15:59:21,031 [INFO] Ordinal encoding cate features
2026-03-08 15:59:21,043 [INFO] Target encoding cate features
100%|██████████| 1/1 [00:00<00:00, 28.91it/s]
2026-03-08 15:59:21,078 [INFO] Start manual binary encoding
100%|██████████| 1/1 [00:00<00:00, 21.10it/s]
2026-03-08 15:59:21,429 [INFO] Filtering and fillna features
100%|██████████| 4/4 [00:00<00:00, 11015.90it/s]
2026-03-08 15:59:21,432 [INFO] Ordinal encoding cate features
2026-03-08 15:59:21,434 [INFO] Target encoding cate features
100%|██████████| 1/1 [00:00<00:00, 357.39it/s]
2026-03-08 15:59:21,437 [INFO] Start manual binary encoding
100%|██████████| 1/1 [00:00<00:00, 22.58it/s]
2026-03-08 15:59:21,779 [INFO] Filtering and fillna features
100%|██████████| 4/4 [00:00<00:00, 9921.48it/s]
2026-03-08 15:59:21,782 [INFO] Ordinal encoding cate features
2026-03-08 15:59:21,784 [INFO] Target encoding cate feature

train_x: (80000, 11)  valid_x: (10000, 11)  test_x: (10000, 11)


## 3. Training (lgb.LGBMRanker.fit)

Pass `train_group` and `valid_group` so LightGBM knows which items belong to the same query. Early stopping monitors NDCG on the validation set.

In [ ]:
ranker = lgb.LGBMRanker(n_estimators=NUM_BOOST_ROUND, **PARAMS)

ranker.fit(
    train_x, train_y.reshape(-1),
    group=train_group,
    eval_set=[(valid_x, valid_y.reshape(-1))],
    eval_group=[valid_group],
    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS)],
)

## 4. Inference and Evaluation (lgb.LGBMRanker.predict)

Score all test items and compute **NDCG@5** and **NDCG@10** averaged across queries. Queries where all labels are 0 (no relevant item) are excluded from the average.

In [6]:
scores = ranker.predict(test_x)

ndcg5_list, ndcg10_list = [], []
start = 0
for g in test_group:
    end = start + g
    y_true  = test_y[start:end].reshape(1, -1)
    y_score = scores[start:end].reshape(1, -1)
    if y_true.shape[1] > 1 and y_true.sum() > 0:  # skip single-item or all-zero queries
        ndcg5_list.append(sklearn_ndcg(y_true, y_score, k=5))
        ndcg10_list.append(sklearn_ndcg(y_true, y_score, k=10))
    start = end

ndcg_at_5  = float(np.mean(ndcg5_list))
ndcg_at_10 = float(np.mean(ndcg10_list))

print(f"NDCG@5:  {ndcg_at_5:.4f}")
print(f"NDCG@10: {ndcg_at_10:.4f}")

NDCG@5:  0.9127
NDCG@10: 0.9353


In [7]:
# Record results for tests - ignore this cell
store_metadata("ndcg_at_5",  ndcg_at_5)
store_metadata("ndcg_at_10", ndcg_at_10)

## 5. Top-K Ranking Example

In a real recommendation pipeline, scores are computed per query (user session) and items are sorted by descending score. Below we display the top-5 items for the first few test queries.

In [8]:
user_ids = test_data[USER_COL].unique()

start = 0
for query_id, (user, g) in enumerate(zip(user_ids[:3], test_group[:3])):
    end = start + g
    query_df = pd.DataFrame({
        ITEM_COL: test_data[ITEM_COL].iloc[start:end].values,
        "score":  scores[start:end],
        "label":  test_y[start:end].reshape(-1),
    })
    top5 = query_df.nlargest(5, "score").reset_index(drop=True)
    print(f"User {user} (rated {g} movies)  top-5:")
    display(top5)
    start = end

User 1 (rated 5 movies)  top-5:


,itemID,score,label
0,172,0.282364,5.0
1,28,-0.230608,4.0
2,152,-0.434589,5.0
3,94,-0.589508,2.0
4,122,-0.670468,3.0


User 2 (rated 2 movies)  top-5:


,itemID,score,label
0,302,0.275522,5.0
1,296,-0.418508,3.0


User 3 (rated 2 movies)  top-5:


,itemID,score,label
0,346,-0.022372,5.0
1,340,-0.053234,5.0


## 6. Save and Load

The underlying booster can be saved with `booster_.save_model()` and restored with `lgb.Booster` for inference, producing identical predictions.

In [ ]:
with TemporaryDirectory() as tmp:
    model_path = os.path.join(tmp, "ranker.lgb")

    ranker.booster_.save_model(model_path)

    booster = lgb.Booster(model_file=model_path)
    loaded_scores = booster.predict(test_x)

assert np.allclose(scores, loaded_scores), "Predictions differ after save/load!"
print("Save/load verified: predictions are identical.")